# Factory Waste Reduction | Gold Analysis

## Business question
**Where is material waste costing the factory the most, and how much could it save by reducing waste?**

This notebook creates the analysis-ready Gold tables used for Tableau and concludes with decision-ready findings.

## 1. Create material-usage fact table

Waste is summarized before joining so multiple waste events do not duplicate a daily-consumption record.

In [ ]:
%sql
CREATE OR REPLACE TABLE gold_material_usage AS
WITH waste_by_usage AS (
  SELECT usage_date, facility, machine_line, shift, sku, source_year, SUM(units_wasted) AS units_wasted
  FROM silver_waste_log
  GROUP BY usage_date, facility, machine_line, shift, sku, source_year
)
SELECT c.usage_date, DATE_FORMAT(c.usage_date, 'yyyy-MM') AS month, c.source_year, c.facility,
       c.machine_line, c.shift, c.sku, p.product_name, p.category, p.unit_of_measure, p.standard_unit_cost,
       c.units_issued, c.units_used, c.units_returned_to_stock, c.units_processed,
       COALESCE(w.units_wasted, 0) AS units_wasted,
       c.units_used + COALESCE(w.units_wasted, 0) AS consumed_units,
       c.units_issued - c.units_used - c.units_returned_to_stock - COALESCE(w.units_wasted, 0) AS balance_variance,
       CAST(COALESCE(w.units_wasted, 0) * p.standard_unit_cost AS DECIMAL(14,2)) AS waste_cost,
       CAST((c.units_used + COALESCE(w.units_wasted, 0)) * p.standard_unit_cost AS DECIMAL(14,2)) AS consumed_cost
FROM silver_daily_consumption c
INNER JOIN silver_product_master p ON c.sku = p.sku
LEFT JOIN waste_by_usage w ON c.usage_date = w.usage_date AND c.facility = w.facility
  AND c.machine_line = w.machine_line AND c.shift = w.shift AND c.sku = w.sku AND c.source_year = w.source_year;

## 2. Create savings scenarios

In [ ]:
%sql
CREATE OR REPLACE TABLE gold_waste_savings_scenarios AS
WITH waste_cost_by_year AS (
  SELECT source_year, SUM(waste_cost) AS total_waste_cost
  FROM gold_material_usage GROUP BY source_year
)
SELECT source_year, scenario_name, reduction_pct, total_waste_cost,
       ROUND(total_waste_cost * reduction_pct, 2) AS potential_savings
FROM waste_cost_by_year
CROSS JOIN (SELECT * FROM VALUES
  ('30% waste reduction', CAST(0.30 AS DECIMAL(3,2))),
  ('50% waste reduction', CAST(0.50 AS DECIMAL(3,2)))
) AS s(scenario_name, reduction_pct);

## 3. Executive KPI view

In [ ]:
%sql
SELECT source_year,
       ROUND(SUM(waste_cost), 2) AS total_waste_cost,
       SUM(units_wasted) AS units_wasted,
       ROUND(100.0 * SUM(waste_cost) / NULLIF(SUM(consumed_cost), 0), 2) AS cost_weighted_waste_rate_pct,
       ROUND(AVG(ABS(balance_variance)), 2) AS avg_daily_balance_variance
FROM gold_material_usage
GROUP BY source_year
ORDER BY source_year;

## 4. Find the biggest waste opportunities

Use this view for a Tableau bar chart: **Waste Cost by Material Category**.

In [ ]:
%sql
SELECT source_year, category, ROUND(SUM(waste_cost), 2) AS waste_cost,
       SUM(units_wasted) AS units_wasted,
       ROUND(100.0 * SUM(waste_cost) / NULLIF(SUM(consumed_cost), 0), 2) AS waste_rate_pct
FROM gold_material_usage
GROUP BY source_year, category
ORDER BY source_year, waste_cost DESC;

## 5. Target operations improvements

Use this for the dashboard heatmap: **Waste Cost by Shift and Machine Line**.

In [ ]:
%sql
SELECT source_year, machine_line, shift, ROUND(SUM(waste_cost), 2) AS waste_cost,
       SUM(units_wasted) AS units_wasted, COUNT(*) AS operating_records
FROM gold_material_usage
GROUP BY source_year, machine_line, shift
ORDER BY source_year, waste_cost DESC;

## 6. Identify root causes

This is the analysis behind a Pareto chart of waste reasons.

In [ ]:
%sql
SELECT w.source_year, w.waste_reason, ROUND(SUM(w.units_wasted * p.standard_unit_cost), 2) AS waste_cost,
       SUM(w.units_wasted) AS units_wasted, COUNT(*) AS waste_events
FROM silver_waste_log w
INNER JOIN silver_product_master p ON w.sku = p.sku
GROUP BY w.source_year, w.waste_reason
ORDER BY w.source_year, waste_cost DESC;

## 7. Scenario recommendation

Start with the 30% scenario as the first operational target. Use the category, line/shift, and reason views above to select the first improvement project rather than spreading effort across every material.

In [ ]:
%sql
SELECT source_year, scenario_name, total_waste_cost, potential_savings
FROM gold_waste_savings_scenarios
ORDER BY source_year, potential_savings;